In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Window

w = Window.partitionBy("call_id").orderBy("_ingested_at")

bronze_schema = StructType([
    StructField("call_id",    StringType(), True),
    StructField("caller",     StringType(), True),
    StructField("callee",     StringType(), True),
    StructField("start_ts",   StringType(), True),
    StructField("duration_s", StringType(), True),
    StructField("status",     StringType(), True),
])

bronze = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/raw/cdrs_dirty.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

silver = (
    bronze
    .withColumn("duration_int",    F.expr("try_cast(duration_s AS INT)"))
    .withColumn("start_ts_parsed", F.expr("try_cast(start_ts AS TIMESTAMP)"))
    .withColumn(
        "_error_type",
        F.when(
            F.col("caller").isNull() |
            (F.trim(F.col("caller")) == "") |
            (F.upper(F.trim(F.col("caller"))) == "NULL"),
            "null_caller",
        )
        .when(
            F.col("start_ts_parsed").isNull(),
            "bad_timestamp",
        )
        .when(
            F.col("duration_int") < 0,
            "negative_duration",
        )
        .when(
            (F.col("status") == "ANSWERED") &
            (F.col("duration_int") == 0),
            "answered_zero",
        )
    )
        .withColumn("_rn", F.row_number().over(w))
    .withColumn(
        "_reject_reason",
        F.coalesce(
            F.col("_error_type"),
            F.when(F.col("_rn") > 1, F.lit("duplicate")),
        ),
    )
)
bronze.write.mode("overwrite").saveAsTable("bronze_cdrs")
display(silver.groupBy("_reject_reason").count().orderBy("_reject_reason"))

dup_ids = silver.groupBy("call_id").count().filter("count > 1").select("call_id")
display(silver.join(dup_ids, "call_id")
              .select("call_id", "_rn", "caller", "duration_s", "_reject_reason")
              .orderBy("call_id", "_rn"))


_reject_reason,count
null,948
answered_zero,14
bad_timestamp,9
duplicate,10
negative_duration,11
null_caller,20


call_id,_rn,caller,duration_s,_reject_reason
22cc5018-5ded-4bb4-bc77-c49f2a2fd7b4,1,+48123451020,681,null
22cc5018-5ded-4bb4-bc77-c49f2a2fd7b4,2,+48123451020,681,duplicate
3835f10f-7fcb-43d5-afce-b6352a39f170,1,+48123451007,1103,null
3835f10f-7fcb-43d5-afce-b6352a39f170,2,+48123451007,1103,duplicate
4dcf1093-05d0-40a5-bd25-8a0cc02419e7,1,+48123451019,0,null
4dcf1093-05d0-40a5-bd25-8a0cc02419e7,2,+48123451019,0,duplicate
544737a1-e10c-4308-846f-02824ca6a68c,1,+48123451007,820,null
544737a1-e10c-4308-846f-02824ca6a68c,2,+48123451007,820,duplicate
6422c39d-698e-4288-8168-0d09fef2845c,1,+48123451024,-1,negative_duration
6422c39d-698e-4288-8168-0d09fef2845c,2,+48123451024,-1,negative_duration


In [0]:
silver = (
    bronze
    .withColumn("duration_int",    F.expr("try_cast(duration_s AS INT)"))
    .withColumn("start_ts_parsed", F.expr("try_cast(start_ts AS TIMESTAMP)"))
    .withColumn(
        "_error_type",
        F.when(
            F.col("caller").isNull() |
            (F.trim(F.col("caller")) == "") |
            (F.upper(F.trim(F.col("caller"))) == "NULL"),
            "null_caller",
        )
        .when(
            F.col("start_ts_parsed").isNull(),
            "bad_timestamp",
        )
        .when(
            F.col("duration_int") < 0,
            "negative_duration",
        )
        .when(
            (F.col("status") == "ANSWERED") &
            (F.col("duration_int") == 0),
            "answered_zero",
        )
    )
        .withColumn("_rn", F.row_number().over(w))
    .withColumn(
        "_reject_reason",
        F.coalesce(
            F.col("_error_type"),
            F.when(F.col("_rn") > 1, F.lit("duplicate")),
        ),
    )
)

In [0]:
silver_clean = (
    silver
    .filter(F.col("_reject_reason").isNull())
    .select(
        "call_id",
        "caller",
        "callee",
        F.col("start_ts_parsed").alias("start_ts"),
        F.col("duration_int").alias("duration_s"),
        "status",
        "_ingested_at",
        "_source_file",
    )
)

In [0]:
bronze_n     = spark.table("bronze_cdrs").count()
silver_n     = spark.table("silver_cdrs").count()
quarantine_n = spark.table("quarantine_cdrs").count()

print(f"bronze={bronze_n}  silver={silver_n}  quarantine={quarantine_n}")

assert bronze_n == silver_n + quarantine_n, (
    f"Row leak: {bronze_n} in, {silver_n + quarantine_n} out"
)
print("Reconciliation OK")

bronze=1012  silver=948  quarantine=64
Reconciliation OK


In [0]:
spark.table("silver_cdrs").printSchema(). start_ts

root
 |-- call_id: string (nullable = true)
 |-- caller: string (nullable = true)
 |-- callee: string (nullable = true)
 |-- start_ts: timestamp (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-7554098047250503>, line 1
----> 1 spark.table("silver_cdrs").printSchema(). start_ts

AttributeError: 'NoneType' object has no attribute 'start_ts'

In [0]:
quarantine = (
    silver
    .filter(F.col("_reject_reason").isNotNull())
    .select(
        "call_id",
        "caller",
        "callee",
        "start_ts",
        "duration_s",
        "status",
        "_ingested_at",
        "_source_file",
        "_reject_reason",
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

quarantine.write.mode("overwrite").saveAsTable("quarantine_cdrs")
print(quarantine.count())


### Reconciliation: quarantine vs injected counts

Two injected duplicates copied rows that were already corrupted. Because validation errors take priority over duplicate detection, both copies are labelled with the original error, which moves 2 rows from duplicate into null_caller and negative_duration. Total quarantined rows (64) match total injected issues.

Total quarantined rows (64) match total injected issues (64).

In [0]:
display(spark.table("quarantine_cdrs").filter(F.col("_reject_reason") == "bad_timestamp"))